## Aim:
How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts

**Idea**\
Test using prompt engineering by passing table of CIGEO pairs to GPT-J model.
Steps:
* Load model and apply it always on one chunk of the document to extract CI failure impacts 
* Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
or 
* Pass dataframe of pairs as input to the model
or
* Use few shot prompting with example answers

**Finally:**
* Compaire all approaches of spatial and temporal linking CI failure impacts

In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
%env PYTORCH_ALLOC_CONF=expandable_segments:True
# %env TORCH_CUDA_ARCH_LIST=8.6

# settings for distributed computing
%env WORLD_SIZE=1
%env RANK=0
%env LOCAL_RANK=0

# NOTE: # WORLD_SIZE: each GPU corresponds to one process (world = no. of processes within a group), processes communicate with each other enabling eg., distributed training
# NOTE: # RANK: IDs of the processes, ranging from 0 up to WORLD_SIZE - 1

In [ ]:
import os
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
from jinja2 import Template
import spacy
import langextract as lx
import textwrap
from langchain_docling import DoclingLoader
from huggingface_hub import login
import torch


sys.path.append("../")
from src.settings import settings as s

torch.manual_seed(42)

# set default location to store model before loading transformers
os.environ["HF_HOME"] = (
    "/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/"
)

## Generate CI_GEO-pairs

In [ ]:
# import spacy_transformers


try:
    nlp = spacy.load(s.settings.SPACY_MODEL)
except (OSError, ValueError):
    print(f"spaCy language model '{s.settings.SPACY_MODEL}' not found. Downloading ...")
    ## loading transformer language model for NER requires additional package
    if (
        s.settings.SPACY_MODEL.endswith("_trf")
        and importlib.util.find_spec("spacy[transformers]") is None
    ):
        !uv add spacy[transformers]
    !uv run python -m spacy download {s.settings.SPACY_MODEL}
    nlp = spacy.load(s.settings.SPACY_MODEL)

print(f"Loaded spaCy language model: {s.settings.SPACY_MODEL}")

In [ ]:
## Create New entity for transport infrastructure and apply it on any doc

## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()


## call nlp model and create pipeline with new entity pattern
# NOTE Creating new entity (CI_TYPE) solves the issue that FAC entities (buildings, airports, highways, bridges, etc.) refer only to the name of the facility (e.g. A76, Ahrtalbahn)
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk("../ner_patterns.jsonl")
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk("../ner_patterns.jsonl")


# store patterns in jsonl file, example:
# ruler.add_patterns([
#     {"label": "CI_TYPE", "pattern": "road?.+"},
#    {"label":"CI_TYPE","pattern":"rail.*$"},
# ])
# ruler.to_disk("../ner_patterns.jsonl")


## load doc
PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"
FILE_PATH = PARSED_TEXT_DIR + "Koks et al 2022 Brief communication_cleaned.md"
loader = DoclingLoader(FILE_PATH)  # use chunks from Docling.Loader
doc = loader.load()

## TODO make as pydantic class with fixed attributes
df_ci_geo = pd.DataFrame(
    columns=[
        "chunk_id",
        "ci_entity",
        "ci_entity_label",
        "geo_entity",
        "geo_entity_label",
        "token_distance",
    ]
)


## get most likely geolocation for each CI entity based on distance
for i, chunk in enumerate(doc):
    nlp_chunk = nlp(chunk.page_content)
    all_ents = [ent for ent in nlp_chunk.ents]
    ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]
    ci_type_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]
    fac_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["FAC"]]

    # check if chunk contains CI_TYPE entities
    if len(ci_type_ents) > 0:
        print(f"\nChunk [{i}], No. CI_TYPE and FAC entities: {len(ci_type_ents)}")
        print(
            f"Contains following entities for CI_TYPE: {ci_type_ents_info}, FAC: {fac_ents_info}"
        )
        print(f"Chunk text [{i}]:", chunk.page_content)
        # print(f"{ {(ci_type_ents[i].text, ci_type_ents[i].label_) for i in range(len(ci_type_ents))} } ")

        # iterate over all entities within chunk
        for ent_idx in range(len(all_ents)):
            # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
            if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                ci_idx = ent_idx

                ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                distance_list = []
                idx_in_chunk = []
                try:
                    for ent_idx in range(len(all_ents)):
                        # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
                        if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
                            geo_idx = ent_idx
                            dist_ent_pair = np.abs(ci_idx - geo_idx)
                            distance_list.append(dist_ent_pair)
                            idx_in_chunk.append((ent_idx))
                            closest_pair_idx = np.argmin(
                                distance_list
                            )  # idx of closest GEO entity
                            distance_closest_pair = distance_list[closest_pair_idx]

                    threshold = 5  # max token distance between CI_TYPE and GEO entity
                    if distance_closest_pair > threshold:
                        print(
                            f""" Token distance between CI_TYPE/FAR and next GEO entity is {distance_closest_pair} and thus larger than the allowed distance of {threshold} tokens """
                        )
                        continue
                    else:
                        print(
                            f"""  Closest GEO entity to CI_TYPE/FAC entity "{all_ents[ci_idx]}" is "{all_ents[idx_in_chunk[closest_pair_idx]]}" at distance {distance_closest_pair}"""
                        )  # TODO constrain min.distance to max value (eg. 5 tokens), issue: likely when distance value is high that geolocation of Ci_type is mentioned in previous sentences or chunk

                    ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                    result_dict = {
                        "chunk_id": i,
                        "ci_entity": all_ents[ci_idx].text,
                        "ci_entity_label": all_ents[ci_idx].label_,
                        "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                        "geo_entity_label": all_ents[
                            idx_in_chunk[closest_pair_idx]
                        ].label_,
                        "token_distance": distance_closest_pair,
                    }
                    df_ci_geo = pd.concat(
                        [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
                    )

                except IndexError:
                    print("No GEO entities found in this chunk.")
                    continue
                # print("\nidx_in_chunk, closest pair idx", idx_in_chunk, closest_pair_idx)

                # spacy.displacy.render(
                #     nlp_chunk, style="ent",
                #     options={"ents": ["CI_TYPE", "GPE", "LOC", "FAC"], "colors": {"CI_TYPE": "violet"}}
                # )

        else:
            print("\nNo CI_TYPE or FAC entities found in this chunk.")
            continue

In [ ]:
df_ci_geo.loc[df_ci_geo["ci_entity"] == "rail"]  # .head(15)

In [ ]:
doc[13].page_content

## LLama with LangExtract

###  Prompt engineering

In [ ]:
question = "Which impacts of infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, societal or economic impacts, the location and possibly the time of the infrastructure failure."

## without s+e impacts
# question = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location and possibly the time of the infrastructure failure."

In [ ]:
# TODO move template to separate file and load via get_template(), define conditions (e.g. user is technical or not)
# TODo make pydantic class model for expected JSON output

# Example code: https://medium.com/@alecgg27895/jinja2-prompting-a-guide-on-using-jinja2-templates-for-prompt-management-in-genai-applications-e36e5c1243cf
# test instead of user_type (see: {% block user_type %}) the modification of question in regard to CI impact types (Tier 1,2,3 and 4 )


## first
#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:

## Text_snippet column
#     In the field "text_snippet" provide the exact linenumbers (within a list) from the context which you used to extract the information about the infrastructure failure and its impacts.
## too long responses


# prompt_template = """

#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:

#     Question: "{{ question }}"

#     Context:
#     {% for item in context %}
#     - {{ item.text }} (Citation: {{ item.citation }})
#     {% endfor %}

#     Evaluate and improve your answer based on the information about critical infrastructure (CI) types (column: "ci_entity") and their geolocations (column: "geo_entity") mentioned in CI locations.

#     CI locations:
#     {% for item in context %}
#     - ("ci_entity" and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "geo_entity"]] }})
#     {% endfor %}


#     In the field "confidence" you should give an estimate how confident you are about the provided information on a scale between 1 (low) to 5 (high).
#     In the "confidence_explanation" field give also a short explanation why you decided for a certain confidence value (maximum two bullet points within a list).


#     Return ONLY valid JSON in the following list format:
#     [{
#         "infrastructure_type": "...",
#         "damage": "...",
#         "location": "...",
#         "time": "...",
#         "duration": "...",
#         "confidence": "...",
#         "confidence_explanation": "[...]"
#     }]

#     Each nested dictionary describes one failure case.
#     DO NOT add commentary or text outside the JSON.


#     Answer:
# """

######################################################################


prompt_template = """

    You are an expert analyst assistant and should use ONLY the provided context to answer the following question:
    
    Question: "{{ question }}"
       
    Context:
    {% for item in context %}
    - {{ item.text }} (Citation: {{ item.citation }})
    {% endfor %}

    
    For the the fields "societal_impact" and "economic_impact" you should try to extract information about societal or economic consequences of infrastructure failures mentioned in the context.
    However, if you do not find any information about societal or economic consequences, then return for these fields a "NAN" value.
    
    For the fields ""infrastructure_type" and "location" you should evaluate and improve your answer based on the information mentioned in CI locations.

    CI locations:
    {% for item in context %}
    - ("ci_entity" and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "geo_entity"]] }})
    {% endfor %}


    In the field "confidence" you should give an estimate how confident you are about the provided information in regard to the societal and economic impacts on a scale between 1 (low) to 5 (high). 
    In the "confidence_explanation" field give also a very short explanation why you decided for a certain confidence value (maximum two bullet points within a list).


    Return ONLY valid JSON in the following list format:
    [{
        "infrastructure_type": "...",
        "damage": "...",
        "societal_impact": "...",
        "economic_impact": "...",
        "location": "...",
        "confidence_explanation": "[...]",
        "confidence": "...",
    }]

    Each nested dictionary describes one failure case.
    DO NOT add commentary or text outside the JSON.
    

    Answer:
"""

template = Template(prompt_template)


# example_context = doc[6].page_content
# chunk_id = 6
# context = [
#     {
#         "text": example_context,
#         "citation": "Koks et al., 2022",
#         "ci_locations": df_ci_geo.loc[df_ci_geo["chunk_id"]==chunk_id],#to_dict(orient="records")
#     },  # TODO use author names or Primary keys from DB
#     # {"text": context, "citation": "Meier et al., 2025"},
# ]

# rendered_prompt = template.render(
#     context=context,
#     question=question,
#     # messages=messages
# )
# print(rendered_prompt)


## left overs
#  Try to be as specific as possible in your answer (bullet points), mention the impacts as numerical information along the location of the impact, and refer to the citations provided in the context.
# # Extract information about infrastructure failures based on the following question:

##  Test Gemini with LangExtract
Langextract does not support Meta models (eg llama) directly (only via Ollama). For this reason we use an alternative for now with an easier implementation: Googles gemini\

**Local models with Ollama**\
Later we can replace Gemini with Mistral, Claude or an local Ollama model eg. via [`result_local = lx.extract(..., model_id="ollama:llama2")`](https://wandb.ai/wandb_fc/genai-research/reports/LangExtract-Transform-text-into-structured-data-with-AI--VmlldzoxNDI1OTMyNw#:~:text=LangExtract%20is%20open%2Dsource%20and,without%20requiring%20any%20fine%2Dtuning.
) 
```
# test with local model from Ollama
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemma2:2b",  # Automatically select Ollama provider
    model_url="http://localhost:8002",  # check if needed 
    fence_output=False,
    use_schema_constraints=False
)
```
Optional: if you use API-based packages eg. fastchat and vLLM for passing HF models to LangExtract. Keep in mind that these packages are needed as LangExtract requires the llm (when from HF, or Llama-2) in an api-like structure (here port: 8001)
See usage examples, https://github.com/google/langextract?utm_source=chatgpt.com

### Basic usage test of LangExtract
Example taken from langExtract github 
set api key for Gemini in `.env`

In [ ]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent(
    """\
    Extract characters, emotions, and relationships in order of appearance.
    Use exact text for extractions. Do not paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context."""
)

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="ROMEO. But soft! What light through yonder window breaks? It is the east, and Juliet is the sun.",
        extractions=[
            lx.data.Extraction(
                extraction_class="character",
                extraction_text="ROMEO",
                attributes={"emotional_state": "wonder"},
            ),
            lx.data.Extraction(
                extraction_class="emotion",
                extraction_text="But soft!",
                attributes={"feeling": "gentle awe"},
            ),
            lx.data.Extraction(
                extraction_class="relationship",
                extraction_text="Juliet is the sun",
                attributes={"type": "metaphor"},
            ),
        ],
    )
]

In [ ]:
# The input text to be processed
input_text = "Lady Juliet gazed longingly at the stars, her heart aching for Romeo"

# Run the extraction
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",
)

In [ ]:
result

In [ ]:
lx.io.save_annotated_documents(
    [result], output_name="extraction_results.jsonl", output_dir="."
)

# Generate the visualization from the file
html_content = lx.visualize("extraction_results.jsonl")
with open("visualization.html", "w") as f:
    if hasattr(html_content, "data"):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

### Prompt and few-shot examples

In [ ]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent(
    """

    Extract from the context information about the flood-affected infrastructure_type, its damage, its location, as well as about 
    the societal or economic impacts which resulted from the infrastructure failure.
    However, if you do not find any information about societal or economic impacts, then return for these fields a "NAN" value.

    Use exact text for extractions. DO NOT paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.

 """
)

# 2. Provide some high-quality examples to guide the model
few_shot_examples = [
    lx.data.ExampleData(
        text="More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR100 million (Hauser, 2021). ",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="motorways",
                attributes={"time": "directly after the event"},
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR100 million",
                attributes={"type": "repair cost"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Of the 112 bridges in the flooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the flood event (MDR, 2021).",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="bridges",
                attributes={"damage type": "destroyed", "number": "62"},
            ),
            lx.data.Extraction(
                extraction_class="location",
                extraction_text="Ahr valley",
                attributes={"region": "Rhineland-Palatinate"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text=" In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Verbond voor Verzekeraars, 2022), "
        "EUR 2.2 billion in Belgium (Assuralia, 2022) and EUR 8.2 billion (GDV, 2022) in Germany. "
        "The event caused major damages to residential and commercial structures and to many critical infrastructure (CI) assets. ",
        extractions=[
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 150 million–EUR 250 million",
                attributes={
                    "type": "insured loss estimates",
                    "location": "Netherlands",
                    "citation": "Verbond voor Verzekeraars, 2022",
                },
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 2.2 billion",
                attributes={
                    "type": "insured loss estimates",
                    "location": "Belgium",
                    "citation": "Assuralia, 2022",
                },
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 8.2 billion",
                attributes={
                    "type": "insured loss estimates",
                    "location": "Germany",
                    "citation": "GDV, 2022",
                },
            ),
        ],
    ),
]

### Appy LangExtract

In [ ]:
context_example = "In mid-July 2021, a persistent low-pressure system caused extreme precipitation in parts of the Belgian, German and Dutch catchments of the Meuse and Rhine rivers. This led to record-breaking water levels and severe ﬂooding (Mohr et al., 2022). Comparable heavy precipitation events in this area have never been registered in most of the affected areas before (Kreienkamp et al., 2021). The German states most affected include Rhineland-Palatinate (Rheinland-Pfalz), with damage to the Ahr River valley (Ahrtal), several regions in\n\nthe Eiffel National Park, to the city of Trier. Flooding in Belgium was concentrated in the Vesdre River valley (districts of Pepinster, Ensival and Verviers), the Meuse River valley (Maaseik, Liége), the Gete River valley (Herk-de-Stad and Halen) and southeast Brussels (Wavre). The Netherlands experienced ﬂooding, mostly concentrated in the southern district of Limburg. In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Verbond voor Verzekeraars, 2022), ∼ EUR 2.2 billion in Belgium (Assuralia, 2022) and ∼ EUR 8.2 billion (GDV, 2022) in Germany. The event caused major damages to residential and commercial structures and to many critical infrastructure (CI) assets. Not only vital functions for ﬁrst responders were affected (e.g. hospitals, ﬁre departments), but also railways, bridges and utility networks (e.g. water and electricity supply) were severely damaged, expecting to take months to years to fully rebuild.\n\nCI is often considered to be the backbone of a well-functioning society (Hall et al., 2016), which is particularly eminent during natural hazards and disasters. For instance, failure of electricity or telecommunication services immediately causes disruptions in the day-to-day functioning of people and businesses, including those outside the directly affected area. Despite the (academic) agreement that failure of infrastructure systems may cause (large-scale) societal disruptions (Garschagen and Sandholz, 2018; Hallegatte et al., 2019; Fekete and Sandholz, 2021), empirical evidence on the impacts of extreme weather events on these systems is still\n\nPublished by Copernicus Publications on behalf of the European Geosciences Union.\n\nE. E. Koks et al.: Flood impacts to infrastructure\n\nlimited. This brief communication provides an overview of the observed ﬂood impacts to large-scale infrastructure systems during the 2021 mid-July western European ﬂood event and how reconstruction of these large-scale systems has progressed. Next, we highlight how some of these observations compare to academic modelling approaches. We conclude with suggestions on moving forward in CI risk modelling, based on the lessons learned from this extreme event.\n\nIn Germany, road and railway infrastructure was severely damaged as documented exemplarily in Fig. 1. Cost estimates reach up to EURO 2 billion Euro (MDR, 2021). More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR 100 million (Hauser, 2021). Of the 112 bridges in the ﬂooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the ﬂood event (MDR, 2021). Over 74 km of roads, paths and bridges in the Ahr valley have been (critically) damaged. In some cases, repairs are expected to take months to years (Zeit Online, 2021). For example, major freeway sections, including parts of the A1 motorway, were closed until early 2022 (24Rhein, 2022). In addition, about 50 000 cars were damaged, causing insurance claims of some EUR 450 million (ADAC, 2021). The German railway provider Deutsche Bahn expects asset damages of around EUR 1.3 billion. Among other things, 180 level crossings, almost 40 signal boxes, over 1000 catenary and signal masts, and 600 km of tracks were destroyed, as well as energy supply systems, elevators and lighting systems (MDR, 2021). As of 11 April 2022, 14 of the affected rail stretches are fully functional again. The less damaged stretches were functional again within 3 months, while some of the most damaged sections in the Ahr valley are expected to be ﬁnished by the end of 2025 (DB, 2022). In Belgium, approximately 10 km of railway tracks and 3000 sleeper tracks have to be replaced; 50 km of catenary needs to be repaired; and 70 000 t of railway track bed needs to be placed, with estimated costs between EUR 30 million–EUR 50 million (Rozendaal, 2021a). Most damages have been repaired within 2 weeks. The most severely damaged railway line (between the villages of Spa and Pepinster) was reopened again on 3 October 2021 (Rozendaal, 2021b). In the Netherlands, no large-scale damage has been reported to transport infrastructure. A few national highways were partly ﬂooded (e.g. the A76 in both directions) or brieﬂy closed (&lt; 3 d) because of the potential of ﬂooding. Most likely due to relative low-ﬂow velocities, damage to Dutch national road infrastructure was limited. Several railway sections were closed (e.g. the rail-\n\nway section between Maastricht and Liége) and some damage occurred to the railway infrastructure, in particular to the electronic “track circuit” devices and saturated railway embankments (Prorail, 2021).\n\nAt the peak of the event, around 200 000 people experienced power outages in Germany. Electricity infrastructure was severely damaged in North Rhine-Westphalia and Rhineland-Palatinate. However, within 2 d around 50 % of the power was restored through repairs and temporary ﬁxes. Within 8 weeks, no emergency power generators were required anymore, with most of the power infrastructure restored in Germany’s affected areas. Some areas, however, only had permanent power infrastructure after 6 months (Westnetz, 2022). The gas distribution network in the Ahr valley was severely damaged. Approximately 133 km of natural gas pipelines, 8500 gas metres, 3400 house pressure regulators, 7220 of the approximately 8000 household connections, and 31 systems measuring and regulating gas pressure have been damaged or destroyed (SWR, 2021). Gas supply was almost fully restored within 4.5 months after the ﬂood event (Energienetze Mittelrhein, 2021). In Belgium, approximately 41 500 people experienced power outages at the peak of the event. This was the result of both damaged and deliberately switched-off electrical cabinets to prevent serious damages. It took around 3 weeks to fully restore power. Similar to Germany, severe damage had been observed to the gas network. In the villages around Liége, such as Chaudfontaine and Pepinster (Belgium), gas supply was fully recovered within 5 months (Grosjean, 2021; De Wolf, 2021). In the Netherlands, 1000– 2000 households experienced a loss of electricity supply at the peak of the event. Between 100 to 200 households had no gas supply. Within several days, electricity supply was restored (Task Force Fact Finding Hoogwater, 2021).\n\nIn the region of Rhineland-Palatinate (Germany), most drinking water supply was restored within 2 months (Hochwasser Ahr, 2021a). However, sewage treatment plants in Altenahr, Mayschoss and Sinzig had been largely destroyed (Hochwasser Ahr, 2021b), and it is expected to take at least 1.5 years to fully repair most sewage treatment plants. Emergency sewage treatment plants have been built in the meantime (GA, 2021). In the Erft region 7 out of 31 wastewater facilities had been destroyed. Many facilities reported pollution of oil and diesel, forming layers up to 15 cm thick (Kuhn, 2021). In addition, much of the groundwater (and soil) in the ﬂood region was mixed with oil (from destroyed residential oil tanks), chemicals such as fertilizers (from wineries and other agriculture) and chemicals from nearby industrial plants. In Sinzig, 3.6 × 106 L of oil–water mixture was recycled, gaining 3600 m3 of oil, to be reused for heating and\n\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\n\nE. E. Koks et al.: Flood impacts to infrastructure\n\nFigure 1. Damage in the Ahr valley, Germany (images taken on 11 August 2021). (a) Destruction of federal highway B266 (A1) and railway (A2) near Heimersheim. (b) Further upstream in the Ahr valley (Altenburg), large stretches of the Ahrtalbahn railway have been destroyed (B1) and the few remaining road and rail bridges show signs of temporary repairs (B2). (c) Riverbed erosion uncovered and destroyed many cables supposed to lie more than 80 cm below surface level (C1) as well as sewers (C2). (d) Inundated electricity distribution infrastructure (D1), road erosion and stabilization (D2), uncovered cables (D3), and collapsed buildings in Schuld. Pictures by Margreet van Marle/Deltares/GEER-association, distributed under Creative Commons Attribution 4.0 license.\n\nindustrial usage (Kuhn, 2021). In the heavily destroyed town of Bad Münstereifel (in the state of North Rhine-Westphalia), drinking water supply was re-established within 5 d after the ﬂood event (most frequently through emergency tanks), and about 50 % of the city centre was reconnected to the freshwater network shortly thereafter however, water had to be boiled before consumption until about 1 month later (Bad Münstereifel, 2021). In Belgium, several towns experienced disruptions in water supply (in particular as a result of pollution). Directly after the event, approximately 3400 families had no access to potable water. Within less than a week, this was reduced to around 1650 families (Terzake, 2021). It took, however, 6 months to rebuild the permanent water supply infrastructure (SWDE, 2022). In the Netherlands, little to no problems have been recorded with regards to water supply.\n\nWe found no information regarding direct impact on solid-waste facilities as a result of the ﬂood event. However, there is a large pressure on the solid-waste sector to clean the affected areas; 1 month after the event, we observed dozens of large temporary waste ﬁlls and frequent incidences of oil pollution in Rhineland-Palatinate during a ﬁeld visit. In the Ahrweiler district alone, the ﬂood caused as much solid waste as normally would be collected over 30 years. In Belgium, the amount of solid waste is estimated around 160 000 t, stored at several places, such as the abandoned highway track A601. This highway has been used for approximately 9 months as a temporary storage for debris (Couplez, 2022). In the Netherlands, there have been primarily problems with waste deposits along the river banks, which is mostly the solid waste transported by the river from further upstream. Thousands of tonnes of tree debris (logs and\n\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\n\nE. E. Koks et al.: Flood impacts to infrastructure\n\nof running water and electricity (Ärzte Zeitung, 2021). After 1.5 months, medical care was guaranteed again in the most affected regions in Rhineland-Palatinate (Hochwasser Ahr, 2021c). In the state of North Rhine-Westphalia, approximately 68 hospitals have been affected, of which several have been affected severely and will take at least 1.5 years to be rebuilt (Fig. 2). Direct damages are estimated to be at least EUR 100 million to repair all medical facilities (Korzilius, 2021). In the town of Eschweiler (Germany), for example, the basement of the hospital was ﬂooded, as well as the outbuildings and the entire outdoor area. The power supply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million. Within 3.5 weeks, the hospital was partly operational, and within 3 months, all hospital operations continued normally (SAH Eschweiler, 2021). The Mutterhaus Ehrang hospital in Trier (Germany) is now permanently closed as the hospital is too severely damaged to rebuild. Furthermore, in the region of Rhineland-Palatinate (Germany), 19 daycare centres and 17 schools suffered damage from the ﬂoods, affecting more than 8000 students (Staib, 2021). Approximately 4 months after the ﬂood event, the district of Bad Neuenahr-Ahrweiler established emergency educational facilities using 297 containers that serve as classrooms, ofﬁces and dining facilities for more than 800 students (Wiesbadener Kurier, 2021). In Belgium, various rural clinics have been affected and were unable to provide any services. Concurrently, in the most affected areas, general-practitioner facilities have been completely destroyed (Le Spécialiste, 2021). In the Netherlands, one nursing home was ﬂooded, and one hospital was evacuated as a precautionary measure.\n\nMost often, large-scale object-based infrastructure impact studies (e.g. Bubeck et al., 2019) only disclose aggregated risk metrics (i.e. country-level risk estimates), which hampers veriﬁcation and validation with observed impacts on smaller scales."

In [ ]:
PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"


filename = "Koks et al 2022 Brief communication_cleaned.md"
filepath = Path(PARSED_TEXT_DIR, filename)
filename_stem = filepath.stem


print(
    f"\n\n ######## -------- Processing document: {filepath.name} -------- ######## \n"
)

## extract authors, publication year and title
citation_pattern = r"(.*?)(\d{4})(.*)"  # split at first occurrence of year
try:
    authors, year, title = re.findall(citation_pattern, filename_stem)[0]
    citation = f"{authors} {year}"
except AttributeError as e:
    print(f"Could not extract citation from title: {e}")
    citation = filename_stem


## load doc
with open(filepath, "r") as file:
    content = file.read()
doc = [lx.data.Document(content)]  # wrap content in Document object


print(
    f"\n  #############  -------- Text-2-Data: {filepath.name} -------- #############  \n"
)


response = lx.extract(
    text_or_documents=context_example,  # doc[0].text,
    prompt_description=prompt,
    examples=few_shot_examples,
    model_id="gemini-2.5-flash",  # "gemini-2.5-pro",
    extraction_passes=1,  # decrease recall but process faster
    max_workers=4,
    max_char_buffer=1000,  # Break big text into small pieces
)
# response = lx.extract(
#     text_or_documents=doc[0].text,
#     prompt_description=prompt,
#     examples=few_shot_examples,
#     model_id="gpt-4o-mini",
#     fence_output=True,              # Required for OpenAI
#     use_schema_constraints=False    # OpenAI doesn't support constraints
# )
print(response)

lx.io.save_annotated_documents(
    [response],
    output_name=f"langextract_{filename_stem}.jsonl",
    output_dir=s.PATH_DATA + "/test_output/",
)

In [ ]:
# Display the response from the previous cell's extraction
response

In [ ]:
# lx.io.save_annotated_documents([result], output_name="../../data/llm_outputs/extraction_results.jsonl", output_dir=".")

# Generate the visualization from the file
html_content = lx.visualize(
    s.PATH_DATA + "/test_output/" + f"langextract_{filename_stem}.jsonl"
)
with open("visualization.html", "w") as f:
    if hasattr(html_content, "data"):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

In [ ]:
?lx.data.Document

In [ ]:
doc[0].tokenized_text.__dir__()

In [ ]:
import gc
import torch

# clean up after each document
gc.collect()
torch.cuda.empty_cache()

### Testing vLLM and FastChat

In [ ]:
# # install before langextract and vllm

# # --tensor-parallel-size 1 --gpu-memory-utilization 0.90
# !python -m vllm.entrypoints.openai.api_server \
#     --model meta-llama/Llama-2-7b-chat-hf \
#     --dtype auto \
#     --port 8000 \
#     --gpu-memory-utilization 0.90


## test with fastchat loading
# documentation: https://fastchat.mintlify.app/install
# uv add fschat[model_worker,webui]

import os


login(token=os.environ["HUGGINGFACE_TOKEN"])  # TODO replace by using pydantic settings

## Import NOTE (cpu/cuda/OOM):
## for 7b we need around 20GB VRAM
# first try to run in with some optimazation (--load-8bit) and to run with CUDA --> make sure to compile against local cuda by running in terminal (adapt cuda version):
# CUDACXX=/usr/local/cuda-13/bin/nvcc CMAKE_ARGS="-DLLAMA_CUBLAS=on -DCMAKE_CUDA_ARCHITECTURES=native" FORCE_CMAKE=1 pip install .
# if GPU still results in OOM --> pin down to cpu only "--device cpu"
# start worker at port 8001, NOTE: set specific port only when default port s already used by other process (eg. docker container - vector db)
# !python3 -m fastchat.serve.controller --port 8004 & !python3 -m fastchat.serve.model_worker --model-path meta-llama/Llama-2-7b-chat-hf  --load-8bit --device cpu --port 8004

# start controller and model worker in temrinal
!python3 -m fastchat.serve.controller --port 8001  & python3 -m fastchat.serve.model_worker --model-path meta-llama/Llama-2-7b-chat-hf --device cpu --port 8005


# !python3 -m fastchat.serve.cli --model-path meta-llama/Llama-2-7b-chat-hf  --load-8bit

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HUGGINGFACE_TOKEN"])

#### response

In [ ]:
safety_df = df_responses.copy()

# save to disk along with prompt text


OUTPUT_DIR = "../" + s.settings.PATH_DATA + "llm_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

outfile_name = "ci_failure_impacts_responses_llama2_with_eco_explicit_3.csv"
outfile_path = OUTPUT_DIR + outfile_name


if not os.path.isfile(outfile_path):
    outfile_response_path = Path(outfile_path)
    outfile_prompt_path = Path(
        OUTPUT_DIR + "prompt_" + outfile_name.replace(".csv", ".txt")
    )

    print(f"Saving prompt and LLm response to {outfile_response_path.parent} ...")
    with open(outfile_prompt_path, "w") as f:
        f.write(prompt_template)
    safety_df.to_csv(outfile_response_path, index=False)
else:
    print(
        f"Output file {Path(outfile_name).stem} already exists. Skip saving to avoid overwriting ..."
    )

In [ ]:
## 10 min


df_ci_geo

## Evaluation